# 5章 構造化データの出力 (2)


### Ollama

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_" -O apple.txt

--2026-10-22 06:17:59--  https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_
Resolving drive.google.com (drive.google.com)... 192.178.219.102, 192.178.219.100, 192.178.219.138, ...
Connecting to drive.google.com (drive.google.com)|192.178.219.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download [following]
--2026-10-22 06:17:59--  https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.217.203.132, 2607:f8b0:400c:c07::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.217.203.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 410321 (401K) [application/octet-stream]
Saving to: ‘apple.txt’

apple.txt           100%[===================>] 400.70K  --.-KB/s

In [ ]:
MAX_LENGTH = 100_000
# サンプルの10-K SEC提出書類からテキストを読み込む
with open('apple.txt', 'r') as file:
    sec_filing = file.read()
sec_filing = sec_filing[:MAX_LENGTH]

In [ ]:
!sudo apt-get update
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Get:

In [ ]:
!pip install ollama

In [ ]:
import subprocess
import time
import os

cmd = "ollama serve"
process = subprocess.Popen(cmd.split(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)

In [ ]:
!ollama pull smollm2:1.7b-instruct-q4_k_m

In [ ]:
BASE_PROMPT = """
You are an expert at structured data extraction.
You will be given unstructured text from an SEC filing.
Extract the mentioned entities and places and return the result
using the specified structure.
""".strip()

from ollama import chat
from pydantic import BaseModel, Field

class SECExtraction(BaseModel):
    mentioned_entities: list[str] = Field(
        description="Company names, executives, investment funds, tickers"
    )
    mentioned_places: list[str] = Field(
        description="Countries, cities, regions, headquarters locations"
    )

OLLAMA_STRUCTURED_OUTPUT_PROMPT_SUFFIX = "Return as JSON."
OLLAMA_STRUCTURED_OUTPUT_TEMPERATURE = 0

def extract_entities_from_sec_filing(doc: str, model: str) -> dict:
    """
    Ollama chatを使用して、SEC提出書類からエンティティと場所を抽出する。

    Args:
        doc: 分析対象となるSEC提出書類のテキスト
        model: 抽出に使用するOllamaモデル

    Returns:
        チャットモデルからの生の応答
    """
    response = chat(
        messages=[
            {
                'role': 'user',
                'content': f"""{BASE_PROMPT}
                {OLLAMA_STRUCTURED_OUTPUT_PROMPT_SUFFIX}

                Document: {doc}"""
            }
        ],
        model=model,
        format=SECExtraction.model_json_schema(),
        # 決定的な出力にするため、0に設定する
        options={'temperature': OLLAMA_STRUCTURED_OUTPUT_TEMPERATURE}
    )
    return response

In [ ]:
doc = sec_filing[:10_000]
model = "smollm2:1.7b-instruct-q4_k_m"

response = extract_entities_from_sec_filing(doc, model)

import json

response_json = json.loads(response.message.content)

In [ ]:
response_json

{'mentioned_entities': ['Apple Inc.'],
 'mentioned_places': ['Fiscal year is the 52- or 53-week period that ends on the last Saturday of September.']}

In [ ]:
print("Extracted entities:", response_json.get('mentioned_entities'))
print("Extracted places:", response_json.get('mentioned_places'))

Extracted entities: ['Apple Inc.']
Extracted places: ['Fiscal year is the 52- or 53-week period that ends on the last Saturday of September.']
